In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import coocurence

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config, 
    X_train, 
    X_val, 
    X_val_seq, 
    ordinal_encoder, 
    return_only_val=False , 
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 15min 13s, sys: 2min 55s, total: 18min 8s
Wall time: 3min 14s


In [5]:
X_train.head()

user,aid
i64,list[i64]
6547648,"[1580108, 474053]"
1854400,"[325324, 350645, ... 1488040]"
670144,"[262425, 1450963, ... 1652922]"
4279040,"[569494, 918969, ... 111356]"
8103296,"[167251, 825464, ... 952459]"


In [6]:
%%time

list_inters = list(X_train.to_pandas()[config.col_aid])

CPU times: user 3.09 s, sys: 470 ms, total: 3.56 s
Wall time: 3.53 s


In [7]:
del X_train
gc.collect()

190

In [13]:
%%time

top_pairs = coocurence.train_cooc_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=20)

extracting max_candidates: 100%|█████| 907899/907899 [00:45<00:00, 19994.08it/s]


CPU times: user 5min 30s, sys: 25 s, total: 5min 55s
Wall time: 5min 56s


In [14]:
%%time

rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)

data_to_predict = X_val_seq.to_pandas()[config.col_train_seq].tolist()
df_preds = rec_cooc.recommend(data_to_predict)

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(df_preds, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_val_preds_{config.num_recs}.parquet')

predicting: 100%|████████████████████| 466825/466825 [00:07<00:00, 58372.11it/s]


CPU times: user 50.2 s, sys: 8.51 s, total: 58.7 s
Wall time: 51.2 s


In [15]:
df_val_preds.head()

index,user,item_id,cooc_score_0,cooc_rank_0
i64,i64,i64,i64,i32
0,13209728,1483947,96,1
0,13209728,1793485,42,2
0,13209728,1410675,32,3
0,13209728,1185660,15,4
0,13209728,1830694,14,5


In [16]:
%%time

recs = (
    df_val_preds.groupby(["index"])
    .agg(
        [
            pl.col("item_id").list(),
        ]
    )
    .sort("index")
    .to_pandas()
)

recs["user"] = X_val_seq.to_pandas()["user"]
recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))
recs = recs.merge(X_val[["user", "target"]].to_pandas())

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

print(utils.recall_several_targets(preds, targets))
print()
print('_' * 10)

  0%|          | 0/443839 [00:00<?, ?it/s]

0.0003957891637883707

__________
CPU times: user 37.9 s, sys: 2.26 s, total: 40.2 s
Wall time: 12.9 s


In [10]:
del data_to_predict, df_preds, df_val_users, df_val_preds, X_val_seq, X_val
gc.collect()

190

In [11]:
del top_pairs
gc.collect()

204

# TEST

In [4]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

CPU times: user 7min 49s, sys: 13min 29s, total: 21min 19s
Wall time: 1min 46s


0

In [5]:
%%time

top_pairs = coocurence.train_cooc_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=20)

extracting max_candidates: 100%|█████| 922584/922584 [00:46<00:00, 20009.24it/s]


CPU times: user 6min 5s, sys: 19.9 s, total: 6min 24s
Wall time: 6min 24s


In [6]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)
df_preds = rec_cooc.recommend(data_to_predict)

df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())
df_test_preds = df_test_users.join(df_preds, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_test_preds_{config.num_recs}.parquet')

predicting: 100%|██████████████████| 1671803/1671803 [00:30<00:00, 55687.97it/s]


CPU times: user 2min 33s, sys: 22.2 s, total: 2min 55s
Wall time: 2min 25s
